## Orchestrator-Workers Workflow
In this workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

### When to use this workflow
This workflow is well-suited for complex tasks where you can't predict the subtasks needed. The key difference from simple parallelization is its flexibility—subtasks aren't pre-defined, but determined by the orchestrator based on the specific input.

In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
from typing import Dict, List, Optional
from util import llm_call, extract_xml

def parse_tasks(tasks_xml: str) -> List[Dict]:
    """Parse XML tasks into a list of task dictionaries."""
    tasks = []
    current_task = {}
    
    for line in tasks_xml.split('\n'):
        line = line.strip()
        if not line:
            continue
            
        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)
    
    return tasks

class FlexibleOrchestrator:
    """Break down tasks and run them in parallel using worker LLMs."""
    
    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
    ):
        """Initialize with prompt templates."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt

    def _format_prompt(self, template: str, **kwargs) -> str:
        """Format a prompt template with variables."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}")

    def process(self, task: str, context: Optional[Dict] = None) -> Dict:
        """Process task by breaking it down and running subtasks in parallel."""
        context = context or {}
        
        # Step 1: Get orchestrator response
        orchestrator_input = self._format_prompt(
            self.orchestrator_prompt,
            task=task,
            **context
        )
        orchestrator_response = llm_call(orchestrator_input)
        print("OUTPUT:", orchestrator_response)
        
        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)
        
        print("\n=== ORCHESTRATOR OUTPUT ===")
        print(f"\nANALYSIS:\n{analysis}")
        print(f"\nTASKS:\n{tasks}")
        
        # Step 2: Process each task
        worker_results = []
        for task_info in tasks:
            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_type=task_info['type'],
                task_description=task_info['description'],
                **context
            )
            
            worker_response = llm_call(worker_input)
            result = extract_xml(worker_response, "response")
            
            worker_results.append({
                "type": task_info["type"],
                "description": task_info["description"],
                "result": result
            })
            
            print(f"\n=== WORKER RESULT ({task_info['type']}) ===\n{result}\n")
        
        return {
            "analysis": analysis,
            "worker_results": worker_results,
        }


### Example Use Case: Marketing Variation Generation



In [12]:
ORCHESTRATOR_PROMPT = """
Analyze this task and break it down into 5-6 distinct approaches:

Task: {task}

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
    <type>formal</type>
    <description>Write a precise, technical version that emphasizes specifications</description>
    </task>
    <task>
    <type>conversational</type>
    <description>Write an engaging, friendly version that connects with readers</description>
    </task>
</tasks>
"""

WORKER_PROMPT = """
Generate content based on:
Task: {original_task}
Style: {task_type}
Guidelines: {task_description}

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"""

In [6]:
import pandas as pd
from util import load_json
data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed/'
df1_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv'
df4_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_4.csv'
df1 = pd.read_csv(df1_path)['file_path'].to_list()
df4 = pd.read_csv(df4_path)['file_path'].to_list()

case_path = data_path + df4[0]
case = load_json(case_path)
case_facts = case['facts']  

In [18]:


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task=F"""Classify the importance of the legal case given the case facts.
      <Facts>
      {case_facts}.
      </Facts>
      
      Your Final Answer: CLASSIFICATION: [NOT KEY CASE or KEY CASE]""",
    context={
        "KEY CASE": 

"""
- Contributes significantly to the development, clarification, or modification of ECHR case law.
- Establishes new legal principles or substantially alters existing ones.
- Addresses unique or emerging societal, legal, or procedural trends that could influence future jurisprudence.
- Has broad implications beyond the immediate case.
""" ,
        "target_audience": "Legal Experts and Lawyers in ECHR Jurisdictions",
        "key_features": ["Key case Identification", "Case Law development"]

}
)

OUTPUT: <analysis>
The task requires determining whether this legal case should be classified as a "key case" that establishes important legal precedent or principles. Several valuable approaches could be used:

1. Precedential Impact Analysis: Examine how this case affects or changes existing legal principles around extradition and life sentences

2. Citation Pattern Analysis: Look at how extensively this case is referenced by subsequent cases and legal documents

3. Legal Principle Establishment: Evaluate whether the case establishes new legal tests or frameworks

4. Jurisdictional Significance: Assess the case's importance within its legal jurisdiction and potential cross-border implications

5. Practical Impact Assessment: Consider the real-world effects on similar future cases and legal practice

6. Historical Context Analysis: Place the case within the broader evolution of relevant legal doctrine
</analysis>

<tasks>
<task>
<type>formal</type>
<description>
Classification: NOT KE

In [19]:
case_path = data_path + df1[1]
case = load_json(case_path)
case_facts = case['facts']  


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task=F"""Classify the importance of the legal case given the case facts.
      <Facts>
      {case_facts}.
      </Facts>
      
      Your Final Answer: CLASSIFICATION: [NOT KEY CASE or KEY CASE]""",
    context={
        "KEY CASE": 

"""
- Contributes significantly to the development, clarification, or modification of ECHR case law.
- Establishes new legal principles or substantially alters existing ones.
- Addresses unique or emerging societal, legal, or procedural trends that could influence future jurisprudence.
- Has broad implications beyond the immediate case.
"""    }
)

OUTPUT: <analysis>
The task requires determining whether this legal case is a landmark/key case or not based on several potential analytical approaches:

1. Precedential Impact Analysis: Examining how this case affects or changes existing legal interpretations and principles

2. Jurisdictional Scope Analysis: Evaluating the case's influence across different legal jurisdictions and systems

3. Legal Principle Development Analysis: Assessing whether the case establishes or significantly modifies important legal principles

4. Procedural Significance Analysis: Examining whether the case introduces new procedural standards or interpretations

5. Contextual Historical Analysis: Evaluating the case's importance within its historical and legal context

Each approach serves different aspects:
- Precedential focuses on future impact
- Jurisdictional examines geographic/systemic reach
- Principle Development looks at substantive legal evolution
- Procedural examines process changes
- Historical 